# Checkpoint 2 FP. Grup 2 - Violin Monica dan Rizki Laely
## Mesin Prediksi & Continual Learning untuk Risk Score

Notebook ini melanjutkan langsung dari Hands-On 1. Di HO1, dataset `features_labels.csv`
dibentuk: satu baris per unit analisis (sel grid 150 m × hari × jam) berisi fitur ruang–waktu
dan label `risk_score` (0–100) hasil *pseudo-labeling* (severity scoring berbasis hukum Illinois,
temporal decay half-life 180 hari, spatial decay Gaussian σ=150 m, dan normalisasi logaritmik).


# 0. Setup

In [ ]:
!pip install -q scikit-learn scipy joblib pandas numpy matplotlib seaborn

In [ ]:
import os, sys, json, warnings, hashlib, shutil, platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn

from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import ks_2samp

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)
np.random.seed(42)
try:                                   # agar simbol Δ/²/· aman dicetak di konsol non-UTF8
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass
print("Setup selesai |", f"sklearn {sklearn.__version__} | pandas {pd.__version__}")

# 1. Load Dataset Fitur + Label dari Hands-On 1

Dataset akhir HO1: 374.790 baris × 20 kolom (19 fitur + 1 label), tanpa nilai kosong.
`risk_score` berdistribusi menyerupai normal (rata-rata ≈ 37,9; median ≈ 38,4; std ≈ 16,9),
konsekuensi dari normalisasi logaritmik di HO1 , menguntungkan untuk regresi.

In [ ]:
DATA_PATH = "features_labels.csv"   # upload file HO1 ke session ini

df = pd.read_csv(DATA_PATH)
print(f"Dataset HO1 dimuat: {df.shape}")
print("Kolom:", list(df.columns))
print("\nNaN:", int(df.isna().sum().sum()), "| cell_id unik:", df["cell_id"].nunique())
df.head()

In [ ]:
df.describe().round(2)

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(df["risk_score"], bins=40, color="#4C72B0")
plt.title("Distribusi Risk Score (label HO1)")
plt.xlabel("Risk Score"); plt.ylabel("Jumlah unit")
plt.tight_layout(); plt.show()

# 2. Baseline Non-ML

Baseline adalah patokan minimal sebelum melatih model apa pun: apakah model ML benar-benar
memberi nilai tambah dibanding aturan sederhana? Selain tiga baseline tutorial (mean global,
per-sel, per-(hari,jam)), saya menambahkan dua baseline kombinasi dengan *fallback* berjenjang
, justru untuk menunjukkan bahwa "lebih kompleks" tidak selalu "lebih baik".

In [ ]:
TARGET_COL = "risk_score"
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True); test_df = test_df.reset_index(drop=True)
print(f"train_df: {train_df.shape} | test_df: {test_df.shape}")

def evaluate(y_true, y_pred, label):
    return {"model": label,
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "R2": r2_score(y_true, y_pred)}

In [ ]:
results = []
gmean = train_df[TARGET_COL].mean()
results.append(evaluate(test_df[TARGET_COL], np.full(len(test_df), gmean), "Baseline: Global Mean"))
cell_map = train_df.groupby("cell_id")[TARGET_COL].mean()
pred_cell = test_df["cell_id"].map(cell_map).fillna(gmean).values
results.append(evaluate(test_df[TARGET_COL], pred_cell, "Baseline: per-Sel"))

hd_map = train_df.groupby(["dow","hour"])[TARGET_COL].mean()
idx = pd.MultiIndex.from_frame(test_df[["dow","hour"]])
p = hd_map.reindex(idx).values; p = np.where(np.isnan(p), gmean, p)
results.append(evaluate(test_df[TARGET_COL], p, "Baseline: per-(hari,jam)"))

ch_map = train_df.groupby(["cell_id","hour"])[TARGET_COL].mean()
idx = pd.MultiIndex.from_frame(test_df[["cell_id","hour"]])
p_ch = ch_map.reindex(idx).values
p = np.where(np.isnan(p_ch), np.where(np.isnan(pred_cell), gmean, pred_cell), p_ch)
results.append(evaluate(test_df[TARGET_COL], p, "Baseline: per-(sel,jam)+fallback"))

cdh_map = train_df.groupby(["cell_id","dow","hour"])[TARGET_COL].mean()
idx = pd.MultiIndex.from_frame(test_df[["cell_id","dow","hour"]])
p_cdh = cdh_map.reindex(idx).values
p = np.where(np.isnan(p_cdh),
            np.where(np.isnan(p_ch), np.where(np.isnan(pred_cell), gmean, pred_cell), p_ch),
            p_cdh)
results.append(evaluate(test_df[TARGET_COL], p, "Baseline: per-(sel,hari,jam)+fallback"))

baseline_results_df = pd.DataFrame(results).round(3)
baseline_results_df

Analisis baseline.
- Baseline per-Sel adalah yang terkuat (MAE ≈ 12,19; R² ≈ 0,198). Ini konsisten dengan temuan
  HO1 bahwa kejahatan terkonsentrasi secara spasial: "lokasi ini secara historis seberapa rawan"
  membawa sinyal terkuat.
- Pola temporal murni sangat lemah (per-(hari,jam), R² ≈ 0,03). Sesuai temuan EDA HO1: distribusi
  antar hari nyaris seragam, dan efek jam baru bermakna ketika berinteraksi dengan lokasi.
- Kombinasi malah memburuk (per-(sel,jam) dan per-(sel,hari,jam) -> R² ≈ −0,04, lebih buruk dari
  mean global). Penyebabnya kelangkaan data: median hanya 12 baris/sel (HO1), tersebar pada
  7×24 = 168 slot (hari,jam), sehingga sebagian besar kombinasi hanya punya <1 baris training ->
  "rata-rata" yang dihafal hanyalah noise satu observasi. Ini pelajaran penting: *granularitas
  lookup table dibatasi oleh kepadatan data, bukan oleh imajinasi kita*. Justru inilah alasan kita
  butuh model yang menggeneralisasi lewat fitur, bukan menghafal kombinasi.
- Strategi fallback: memakai rantai (sel,hari,jam) -> (sel,jam) -> (sel) -> global. Namun karena
  level terhalus terlalu langka, fallback praktis selalu jatuh ke level (sel). Kesimpulannya, per-Sel
  sudah menjadi *fallback* yang paling stabil , dan itulah baseline yang saya jadikan pembanding utama.

# 3. Feature Vector Assembly 

## 3.1 Audit kebocoran: tiga jalur, bukan satu

Versi sebelumnya hanya menjaga satu jalur kebocoran (target encoding) dan menganggap sisanya aman.
Audit ulang menemukan tiga jalur, dan dua di antaranya masih terbuka:

| # | Jalur kebocoran | Mengapa bocor | Status di versi lama | Penanganan sekarang |
| :- | :-- | :-- | :-- | :-- |
| 1 | Agregat per-sel HO1 (`n_crimes`, `density_ratio`, `mean_severity`, `max_severity`, `pct_violent`, `n_distinct_types`) | Dihitung di HO1 dari **seluruh** dataset , termasuk baris yang kelak jadi holdout. Baris test ikut membentuk fitur baris train, dan sebaliknya. | **Bocor** (kolom dipakai apa adanya) | Dibentuk ulang di `.fit()` **hanya dari baris fit-set** |
| 2 | Target encoding `cell_id` | Statistik memang dari train, tetapi baris train di-encode memakai peta yang **memuat label barisnya sendiri**. *Smoothing* meredam, tidak menghapus. | **Bocor** (in-fold encoding) | *Out-of-fold* encoding (KFold=5) untuk baris fit; peta penuh hanya untuk data lain |
| 3 | `crime_count` pada baris yang sama | Jumlah kejadian di slot (sel×hari×jam) yang justru dipakai HO1 untuk membentuk `risk_score` slot itu. Saat *serving* (memprediksi slot yang belum terjadi) angka ini belum ada. | Tidak dibahas | Tetap dipakai (ikut definisi tutorial), tapi **diukur** lewat ablasi dan dicatat sebagai batas serving |

Prinsip yang sekarang dipegang: **setiap statistik lintas-baris harus lahir di `.fit()`, dan `.fit()`
hanya boleh melihat data training.** `.transform()` tidak menghitung apa pun , ia hanya memetakan.
Konsekuensinya representasi saat training identik dengan saat serving, dan objek assembler ikut
disimpan bersama model di checkpoint (lihat Bagian 9).

## 3.2 Yang bisa dan tidak bisa dibentuk ulang dari file ini

- Bisa (bersih total): fitur volume: `n_crimes_fit` = `sum(crime_count)` per sel atas baris
  fit-set, dan `density_ratio_fit` = normalisasinya terhadap rata-rata kota versi fit-set. Namanya
  sengaja dibedakan dari kolom HO1: ini fitur milik kita , *"volume kejahatan sel ini menurut data
  yang sudah tersedia"* , yang sah dibentuk tanpa menyentuh baris holdout. Apakah ia mengukur
  konstruk yang sama dengan `n_crimes` HO1 diperiksa langsung di sel berikutnya, bukan diasumsikan.
- Tidak bisa sepenuhnya: profil severity: `mean_severity`, `max_severity`, `pct_violent`,
  `n_distinct_types` adalah konstanta per-sel yang di HO1 diringkas dari *catatan kejadian*
  (severity per kejadian), dan kolom severity per-baris itu sudah tidak ada di
  `features_labels.csv`. Yang bisa dilakukan di sini: memperlakukannya sebagai *lookup* yang
  di-*fit* dari train (menutup kebocoran "sel tak dikenal"), lalu **mengukur harga kebersihan
  penuh** dengan menjalankan varian yang membuang keempatnya (baris "tanpa agregat severity" pada
  tabel Bagian 4). Perbaikan tuntas menuntut agregasi ulang di HO1 *sebelum* split , dicatat di
  Bagian 10.

Keputusan fitur lain tetap sama seperti sebelumnya dan tetap berbasis bukti: 6 agregat spasial HO1
dipertahankan sebagai konsep (korelasinya ke `risk_score` 0,18–0,35 vs `lon_r` 0,12), dan `cell_id`
mentah tidak pernah masuk model , diwakili `lat_r/lon_r` + `te_cell`.

In [ ]:
BASE_COLS     = ["lat_r","lon_r","hour_sin","hour_cos","dow_sin","dow_cos","crime_count"]  # 7 fitur tutorial
VOLUME_HO1    = ["n_crimes", "density_ratio"]                    # kolom HO1 (dihitung dari SELURUH data)
VOLUME_REFIT  = ["n_crimes_fit", "density_ratio_fit"]            # versi kita: dari fit-set saja
SEVERITY_AGG  = ["mean_severity","max_severity","pct_violent","n_distinct_types"]  # lookup dari fit-set


class FeatureAssembler:
    """Perakit feature vector yang aman dari kebocoran.

    Kontrak:
      * `fit`/`fit_transform` HANYA boleh diberi data training.
      * `transform` tidak menghitung statistik apa pun , hanya memetakan hasil fit,
        sehingga representasi saat training == saat serving.
      * objek ini disimpan bersama model di setiap checkpoint (lihat Bagian 9).

    Catatan penamaan: pada mode "refit" fitur volume diberi nama `n_crimes_fit` /
    `density_ratio_fit`, bukan `n_crimes` / `density_ratio`. Keduanya memang bukan kolom HO1:
    ini fitur milik kita sendiri , "volume kejahatan sel ini menurut data yang sudah tersedia" ,
    yang sah dibentuk tanpa menyentuh baris holdout. Kedekatannya dengan kolom HO1 diperiksa
    di sel berikutnya.

    Parameter penting
      agg_source : "refit"  -> agregat per-sel dihitung ulang dari fit-set (leak-safe)
                   "ho1_raw"-> pakai kolom HO1 apa adanya (BOCOR; hanya untuk pembanding)
      use_te     : target encoding cell_id, out-of-fold pada fit-set (leak-safe)
    """

    def __init__(self, use_is_weekend=True, use_crime_count=True,
                 use_volume_agg=True, use_severity_agg=True,
                 agg_source="refit", use_te=False,
                 te_m=20, te_folds=5, random_state=42):
        assert agg_source in ("refit", "ho1_raw")
        self.use_is_weekend   = use_is_weekend
        self.use_crime_count  = use_crime_count
        self.use_volume_agg   = use_volume_agg
        self.use_severity_agg = use_severity_agg
        self.agg_source       = agg_source
        self.use_te           = use_te
        self.te_m, self.te_folds, self.random_state = te_m, te_folds, random_state

    # ---------- daftar kolom ----------
    def _row_cols(self):
        cols = [c for c in BASE_COLS if c != "crime_count" or self.use_crime_count]
        return cols + (["is_weekend"] if self.use_is_weekend else [])

    def _volume_cols(self):
        if not self.use_volume_agg:
            return []
        return VOLUME_HO1 if self.agg_source == "ho1_raw" else VOLUME_REFIT

    def _cell_cols(self):
        return self._volume_cols() + (SEVERITY_AGG if self.use_severity_agg else [])

    # ---------- fit ----------
    def fit(self, data):
        """Semua statistik lintas-baris lahir di sini , dan hanya dari `data`."""
        self.global_target_ = float(data[TARGET_COL].mean())
        g = data.groupby("cell_id")

        # (1) volume: DIHITUNG ULANG dari baris fit-set saja.
        #     density_ratio_fit ternormalisasi terhadap rata-rata kota versi fit-set,
        #     jadi perbedaan skala akibat fit-set yang lebih kecil ikut ternetralkan.
        n_crimes_fit = g["crime_count"].sum()
        stats = pd.DataFrame(index=n_crimes_fit.index)
        stats["n_crimes_fit"]      = n_crimes_fit
        stats["density_ratio_fit"] = n_crimes_fit / n_crimes_fit.mean()
        stats["rows_per_cell"]     = g.size()

        # (2) profil severity: konstanta per-sel warisan HO1 -> diambil sebagai lookup fit-set.
        #     Menutup kebocoran "sel tak dikenal"; residu kebocoran tingkat-kejadian
        #     didokumentasikan di Bagian 3.2 dan diukur lewat varian use_severity_agg=False.
        for c in SEVERITY_AGG:
            stats[c] = g[c].mean()

        self.cell_stats_ = stats
        self.fallback_   = stats.median()          # sel yang tak pernah terlihat -> median kota
        self.n_fit_rows_, self.n_fit_cells_ = len(data), len(stats)

        # (3) target encoding dgn smoothing (peta penuh; hanya untuk data NON-fit)
        if self.use_te:
            agg = g[TARGET_COL].agg(["mean", "count"])
            self.te_map_ = ((agg["mean"] * agg["count"] + self.global_target_ * self.te_m)
                            / (agg["count"] + self.te_m))

        self.feature_names_ = self._row_cols() + self._cell_cols() + (["te_cell"] if self.use_te else [])
        return self

    # ---------- transform ----------
    def _assemble(self, data):
        X = data[self._row_cols()].copy()
        want = self._cell_cols()
        if want:
            if self.agg_source == "ho1_raw":
                X[want] = data[want].values          # <-- jalur BOCOR, sengaja untuk pembanding
            else:
                m = self.cell_stats_.reindex(data["cell_id"].values)[want]
                m.index = X.index
                X[want] = m.fillna(self.fallback_[want])
        return X

    def transform(self, data):
        self._check_fitted()
        X = self._assemble(data)
        if self.use_te:
            X["te_cell"] = data["cell_id"].map(self.te_map_).fillna(self.global_target_).values
        return X[self.feature_names_]

    def fit_transform(self, data):
        """Untuk data fit, te_cell dihitung OUT-OF-FOLD agar baris tidak melihat labelnya sendiri."""
        self.fit(data)
        X = self._assemble(data)
        if self.use_te:
            X["te_cell"] = self._oof_te(data)
        return X[self.feature_names_]

    def _oof_te(self, data):
        y, cells = data[TARGET_COL].values, data["cell_id"].values
        oof = np.empty(len(data), dtype=float)
        kf = KFold(n_splits=self.te_folds, shuffle=True, random_state=self.random_state)
        for tr, va in kf.split(data):
            gm = y[tr].mean()
            s = pd.DataFrame({"c": cells[tr], "y": y[tr]}).groupby("c")["y"].agg(["mean", "count"])
            m = (s["mean"] * s["count"] + gm * self.te_m) / (s["count"] + self.te_m)
            oof[va] = pd.Series(cells[va]).map(m).fillna(gm).values
        return oof

    def _check_fitted(self):
        if not hasattr(self, "feature_names_"):
            raise RuntimeError("FeatureAssembler belum di-fit.")

    def config(self):
        return {"use_is_weekend": self.use_is_weekend, "use_crime_count": self.use_crime_count,
                "use_volume_agg": self.use_volume_agg, "use_severity_agg": self.use_severity_agg,
                "agg_source": self.agg_source, "use_te": self.use_te,
                "te_m": self.te_m, "te_folds": self.te_folds}


y_train, y_test = train_df[TARGET_COL].values, test_df[TARGET_COL].values
asm = FeatureAssembler()
X_train = asm.fit_transform(train_df)
X_test  = asm.transform(test_df)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"assembler di-fit dari {asm.n_fit_rows_:,} baris / {asm.n_fit_cells_:,} sel train")
print("fitur:", asm.feature_names_)
X_train.head()

In [ ]:
recipe = pd.DataFrame({
    "ho1_n_crimes":   df.groupby("cell_id")["n_crimes"].first(),
    "sum_crime_count": df.groupby("cell_id")["crime_count"].sum(),
})
rho = recipe.corr(method="spearman").iloc[0, 1]
print("--- Apakah n_crimes HO1 = jumlah crime_count per sel? (seluruh data) ---")
print(f"sel yang identik persis   : {(recipe['ho1_n_crimes'] == recipe['sum_crime_count']).mean():.1%}")
print(f"korelasi Spearman         : {rho:.4f}")
print(f"rasio rata-rata jumlah/HO1: {(recipe['sum_crime_count']/recipe['ho1_n_crimes']).mean():.3f}")
print("-> " + ("konstruk SAMA: n_crimes_fit adalah versi train-only dari n_crimes HO1.\n"
                if rho > 0.95 else
                "konstruk BERBEDA: n_crimes_fit tetap fitur volume yang sah, tetapi bukan "
                "pengganti n_crimes HO1. Bandingkan dengan varian use_volume_agg=False.\n"))

chk = pd.DataFrame({
    "ho1_raw":    test_df["density_ratio"].values,           
    "train_only": X_test["density_ratio_fit"].values,        
})
unseen = float((~test_df["cell_id"].isin(asm.cell_stats_.index)).mean())
print("--- density_ratio HO1 vs density_ratio_fit (train-only) di holdout ---")
print(f"korelasi Pearson           : {chk.corr().iloc[0,1]:.4f}")
print(f"korelasi Spearman (urutan) : {chk.corr(method='spearman').iloc[0,1]:.4f}")
print(f"rata-rata |selisih relatif|: {(np.abs(chk.ho1_raw - chk.train_only)/chk.ho1_raw).mean():.2%}")
print(f"baris holdout dari sel yang TAK ADA di train: {unseen:.3%} -> fallback median kota\n")

asm_te    = FeatureAssembler(use_te=True)
te_oof    = asm_te.fit_transform(train_df)["te_cell"].values                                   
te_infold = train_df["cell_id"].map(asm_te.te_map_).fillna(asm_te.global_target_).values        
print("--- te_cell pada baris TRAIN: korelasi terhadap label (makin tinggi = makin bocor) ---")
print(f"in-fold (versi lama)        : {np.corrcoef(te_infold, y_train)[0,1]:.4f}")
print(f"out-of-fold (sekarang)      : {np.corrcoef(te_oof,    y_train)[0,1]:.4f}")
print(f"OOF di HOLDOUT (acuan jujur): "
      f"{np.corrcoef(asm_te.transform(test_df)['te_cell'].values, y_test)[0,1]:.4f}")
print("Selisih in-fold vs holdout adalah optimisme yang dulu ikut masuk ke skor model linear.")

# 4. Model Regresi: Training & Pemilihan

`risk_score` kontinu 0–100 -> regresi. Model dilatih bertahap agar efek *feature assembly* dapat
dipisahkan dari efek kelas model. Semua varian memakai `FeatureAssembler` yang di-*fit* ulang pada
`train_df` saja, jadi setiap angka di bawah adalah angka leak-safe:

1. `LinearRegression`, 7 fitur baseline tutorial (titik awal),
2. `LinearRegression`, 14 fitur (efek feature assembly),
3. `LinearRegression`, 14 fitur + `te_cell` **out-of-fold** (efek encoding lokasi),
4. `RandomForestRegressor` dan `HistGradientBoostingRegressor` (non-linearitas & interaksi
   lokasi×waktu yang ditemukan di EDA HO1),
5. dua **ablasi audit kebocoran**: tanpa agregat severity (10 fitur, bersih total) dan tanpa
   `crime_count` (13 fitur, meniru kondisi serving),
6. satu baris **pembanding bocor**: agregat HO1 dipakai apa adanya , persis konfigurasi versi
   sebelumnya. Selisihnya terhadap baris leak-safe adalah besarnya optimisme yang tadinya
   tidak disadari.

In [ ]:
def run(model_factory, label, **asm_kw):
    """Satu eksperimen: assembler di-fit pada train saja, lalu dievaluasi di holdout."""
    a = FeatureAssembler(**asm_kw)
    Xtr = a.fit_transform(train_df)          
    Xte = a.transform(test_df)               
    m = model_factory(); m.fit(Xtr, y_train)
    r = evaluate(y_test, m.predict(Xte), label)
    r["n_fitur"] = Xtr.shape[1]
    return r, m, a

HGB = lambda: HistGradientBoostingRegressor(max_iter=400, learning_rate=0.08, random_state=42)
RF  = lambda: RandomForestRegressor(n_estimators=100, n_jobs=-1, min_samples_leaf=3, random_state=42)

model_results = []
def add(factory, label, **kw):
    r, m, a = run(factory, label, **kw); model_results.append(r); return m, a

add(LinearRegression, "LinReg (7 fitur tutorial)",
    use_is_weekend=False, use_volume_agg=False, use_severity_agg=False)
add(LinearRegression, "LinReg (14 fitur)")
add(LinearRegression, "LinReg (14 fitur + te_cell OOF)", use_te=True)
rf,  asm_rf  = add(RF,  "RandomForest (14 fitur)")
hgb, asm_hgb = add(HGB, "HistGB (14 fitur)")
add(HGB, "HistGB (14 fitur + te_cell OOF)", use_te=True)
add(HGB, "HistGB (10 fitur, tanpa agregat severity)", use_severity_agg=False)
add(HGB, "HistGB (13 fitur, tanpa crime_count)",      use_crime_count=False)
add(HGB, "[BOCOR] HistGB 14 fitur, agregat HO1 apa adanya", agg_source="ho1_raw")

pd.DataFrame(model_results)[["model","n_fitur","MAE","RMSE","R2"]].round(3)

In [ ]:
imp = pd.Series(rf.feature_importances_, index=asm_rf.feature_names_).sort_values()
plt.figure(figsize=(7,5))
imp.plot(kind="barh", color="#55A868")
plt.title("Feature Importance (Random Forest, fitur leak-safe)")
plt.xlabel("Importance"); plt.tight_layout(); plt.show()
imp.sort_values(ascending=False).round(3)

Analisis model (angka pada tabel di atas; sel berikutnya mencetak selisih pentingnya).

- Feature assembly tetap pengungkit terbesar: Menambah 6 agregat spasial HO1 ke model linear
  menaikkan R² dari level "kalah baseline per-Sel" ke level "menyamai baseline" , bukti kuantitatif
  bahwa fitur yang dibuang tutorial justru yang paling berharga. Kesimpulan ini bertahan setelah
  agregatnya direkonstruksi dari train saja, jadi ia bukan artefak kebocoran.
- Pohon tetap tidak butuh `te_cell`: Identitas sel sudah tertangkap lewat `lat_r/lon_r` +
  agregat spasial, sehingga `te_cell` hanya menambah redundansi. Keputusan tidak berubah:
  te_cell untuk model linear, tidak untuk model pohon.
- Harga kebersihan: Baris "tanpa agregat severity" adalah varian yang 100% bebas kebocoran dari
  file ini; selisihnya terhadap varian 14 fitur adalah biaya kebersihan penuh. Baris
  "tanpa `crime_count`" menunjukkan performa pada kondisi *serving* sesungguhnya , saat jumlah
  kejadian di slot yang diprediksi belum diketahui. Keduanya dilaporkan apa adanya, bukan
  disembunyikan di balik angka terbaik.
- Besar optimisme versi lama: terbaca dari selisih baris `[BOCOR]` terhadap baris leak-safe yang
  setara. Semua angka lain di notebook ini memakai jalur leak-safe.
- Batas atas performa jujur: R² tetap tertahan di kisaran ~0,3: `risk_score` HO1 memuat
  kontribusi spatial decay dari sel tetangga dan *temporal decay* per kejadian yang tidak dapat
  direkonstruksi dari agregat per-sel. Ada noise ireduksibel , temuan valid, bukan kegagalan.

In [ ]:
res = pd.DataFrame(model_results).set_index("model")
def d(a, b, m="MAE"):
    return res.loc[a, m] - res.loc[b, m]

print("--- Efek feature assembly (model linear) ---")
print(f"MAE 7 fitur -> 14 fitur           : {d('LinReg (7 fitur tutorial)','LinReg (14 fitur)'):+.3f} "
      f"(negatif = 14 fitur lebih buruk)")
print("\n--- Efek target encoding ---")
print(f"LinReg: 14 fitur -> +te_cell OOF  : {d('LinReg (14 fitur)','LinReg (14 fitur + te_cell OOF)'):+.3f}")
print(f"HistGB: 14 fitur -> +te_cell OOF  : {d('HistGB (14 fitur)','HistGB (14 fitur + te_cell OOF)'):+.3f}")
print("\n--- Harga kebersihan / batas serving (HistGB) ---")
print(f"biaya membuang agregat severity   : "
      f"{d('HistGB (10 fitur, tanpa agregat severity)','HistGB (14 fitur)'):+.3f} MAE")
print(f"biaya kehilangan crime_count      : "
      f"{d('HistGB (13 fitur, tanpa crime_count)','HistGB (14 fitur)'):+.3f} MAE")
print("\n--- Besar optimisme akibat kebocoran versi lama ---")
gap = d("HistGB (14 fitur)", "[BOCOR] HistGB 14 fitur, agregat HO1 apa adanya")
print(f"MAE leak-safe - MAE bocor         : {gap:+.3f} "
      f"({'versi lama terlalu optimis' if gap > 0 else 'selisih dapat diabaikan'})")

# 5. Evaluasi Model vs Baseline

Model terbaik (HistGB) dievaluasi di `test_df` yang tak pernah dilihat saat training, dibandingkan
eksplisit dengan baseline terkuat.

In [ ]:
best_model, asm_best = hgb, asm_hgb
X_test = asm_best.transform(test_df)          

final_results_df = pd.DataFrame(
    results + [evaluate(y_test, best_model.predict(X_test), "Model: HistGradientBoosting (leak-safe)")]
).round(3)
final_results_df

In [ ]:
plt.figure(figsize=(9,4))
order = final_results_df.sort_values("MAE", ascending=False)
sns.barplot(data=order, x="model", y="MAE", hue="model", palette="rocket", legend=False)
plt.title("Perbandingan MAE: Baseline vs Model (makin kecil makin baik)")
plt.ylabel("MAE"); plt.xlabel(""); plt.xticks(rotation=25, ha="right")
plt.tight_layout(); plt.show()

In [ ]:
pred = best_model.predict(X_test)
resid = y_test - pred

fig, ax = plt.subplots(1, 2, figsize=(12,5))
ax[0].scatter(y_test, pred, alpha=0.15, s=8, color="#4C72B0")
ax[0].plot([0,100],[0,100],"--",color="gray")
ax[0].set_xlabel("Risk Score Aktual"); ax[0].set_ylabel("Prediksi")
ax[0].set_title("Prediksi vs Aktual , HistGB")

band = pd.cut(y_test, [0,20,40,60,80,100])
mae_band = pd.Series(np.abs(resid)).groupby(band).mean()
mae_band.plot(kind="bar", ax=ax[1], color="#C44E52")
ax[1].set_title("MAE per band Risk Score"); ax[1].set_ylabel("MAE"); ax[1].set_xlabel("band")
plt.tight_layout(); plt.show()
mae_band.round(2)

Analisis evaluasi.
- Model tetap mengalahkan seluruh baseline pada MAE, RMSE, dan R² sekaligus , dan kini
  perbandingannya adil, karena fitur model tidak lagi meminjam informasi dari baris holdout.
  Keunggulannya nyata namun tidak dramatis, konsisten dengan noise ireduksibel di atas.
- Residual tidak bias (mean ≈ 0) tetapi errornya berbentuk-U terhadap band risiko: MAE terkecil
  di band tengah dan membesar di kedua ekstrem. Ini *regression to the mean* , model menarik
  prediksi ke pusat distribusi, sehingga hotspot ekstrem *under-predicted* dan sel sangat aman
  *over-predicted*.
- Implikasi untuk sistem risk-score: karena keputusan operasional menyasar hotspot (band atas),
  MAE global yang bagus bisa menyesatkan. Untuk konteks ini MAE lebih relevan daripada RMSE
  (RMSE terlalu didominasi ekor yang paling *noisy*), tetapi keduanya wajib dilengkapi error
  per-band agar performa di hotspot terpantau eksplisit , dan metrik per-band inilah yang ikut
  dicatat ke registry pada setiap versi (Bagian 9).

# 6. Continual Learning , Simulasi Kedatangan Data

Keterbatasan yang jujur (sama seperti catatan tutorial): `features_labels.csv` sudah diagregasi
per (sel×hari×jam); kolom `Datetime` per-kejadian sudah melebur dan tidak ada di file ini.
Maka batch berdasarkan urutan waktu asli tidak dapat dibentuk dari file ini.

Tutorial mengatasinya dengan mengacak lalu menyuntik drift buatan (mengalikan `crime_count` dan
`risk_score`). Saya menilai itu kurang jujur , drift-nya fiktif. Sebagai gantinya saya membandingkan
tiga skema *batching* dan memilih yang menghasilkan drift nyata dari data itu sendiri:

1. random , acak (seperti tutorial, tanpa injeksi). Menghasilkan batch i.i.d.: tidak ada drift.
2. spatial , rollout geografis barat->timur (`gx`). Drift ada tapi moderat.
3. coverage-expansion (density) , sistem di-*pilot* pada area sepi dulu lalu diperluas ke area
   padat (`density_ratio` menaik). Ini skenario deployment MLOps yang realistis dan menghasilkan
   drift kovariat dan label yang nyata , skema yang saya pakai.

Batch dipisah dari `train_df`; `holdout_df` (= `test_df`) dibekukan dan dipakai konsisten untuk semua
versi agar perbandingan antar-model adil.

Catatan soal kebocoran: pengurutan batch memakai kolom `density_ratio` **asli HO1**. Itu keputusan
*desain simulasi* (menentukan urutan kedatangan data, seperti jadwal rollout yang memang diketahui
operator), bukan fitur yang masuk ke model. Fitur yang dipakai model tetap direkonstruksi ulang dari
data yang sudah tiba pada setiap checkpoint , lihat Bagian 8.

In [ ]:
holdout_df = test_df.copy()
stream_df = train_df.copy()
N_BATCHES = 5

def make_batches(scheme, n=N_BATCHES):
    s = stream_df.copy()
    if scheme == "random":
        s = s.sample(frac=1.0, random_state=7).reset_index(drop=True)
    elif scheme == "spatial":
        s = s.sort_values("gx").reset_index(drop=True)
    elif scheme == "density":
        s = s.sort_values("density_ratio").reset_index(drop=True)
    bs = len(s)//n
    return [s.iloc[i*bs:(len(s) if i==n-1 else (i+1)*bs)].reset_index(drop=True) for i in range(n)]

batches = make_batches("density")
for i,b in enumerate(batches):
    print(f"Batch {i}: n={len(b)} | density_ratio "
          f"[{b['density_ratio'].min():.2f}, {b['density_ratio'].max():.2f}] "
          f"| risk_mean={b['risk_score'].mean():.1f}")

# 7. Drift Detection , KS + PSI

Baseline tutorial hanya memakai uji KS univariat dengan satu ambang p-value. Masalahnya, pada
sampel besar (~60 rb/baris) KS terlalu sensitif: perbedaan sekecil apa pun menjadi "signifikan"
(p->0), sehingga selalu memicu retrain. Saya menggabungkannya dengan PSI (Population Stability
Index) yang mengukur magnitudo pergeseran distribusi.

Drift dinyatakan hanya jika KEDUA syarat terpenuhi: `p_value(KS) < 0,01` dan `PSI > 0,2`.
Artinya pergeseran harus signifikan secara statistik sekaligus cukup besar untuk penting.
Konvensi PSI industri: <0,1 stabil, 0,1–0,2 moderat, >0,2 signifikan.

In [ ]:
DRIFT_COLS = ["crime_count", "n_distinct_types", "mean_severity", "density_ratio", "risk_score"]
KS_ALPHA, PSI_THRESH = 0.01, 0.20

def psi(ref, cur, bins=10):
    qs = np.quantile(ref, np.linspace(0,1,bins+1)); qs[0]=-np.inf; qs[-1]=np.inf
    r = np.clip(np.histogram(ref, qs)[0]/len(ref), 1e-4, None)
    c = np.clip(np.histogram(cur, qs)[0]/len(cur), 1e-4, None)
    return float(np.sum((c-r)*np.log(c/r)))

def detect_drift(reference_df, new_batch_df, columns=DRIFT_COLS,
                 alpha=KS_ALPHA, psi_thresh=PSI_THRESH):
    report, drift_detected = {}, False
    for col in columns:
        _, p_value = ks_2samp(reference_df[col], new_batch_df[col])
        psi_val = psi(reference_df[col].values, new_batch_df[col].values)
        is_drift = bool(p_value < alpha and psi_val > psi_thresh)
        report[col] = {"ks_p": float(p_value), "psi": psi_val, "drift": is_drift}
        if is_drift: drift_detected = True
    return drift_detected, report

In [ ]:
rows = []
for scheme in ["random","spatial","density"]:
    bs = make_batches(scheme)
    flag, rep = detect_drift(bs[0], bs[-1])   
    for col, d in rep.items():
        rows.append({"scheme": scheme, "kolom": col,
                     "KS_p": round(d["ks_p"],4), "PSI": round(d["psi"],3),
                     "drift": d["drift"]})
pd.DataFrame(rows).pivot_table(index="kolom", columns="scheme",
        values="PSI").round(3)

In [ ]:
for scheme in ["random","spatial","density"]:
    bs = make_batches(scheme)
    flag,_ = detect_drift(bs[0], bs[-1])
    print(f"{scheme:8s}: drift batch0 vs batch{N_BATCHES-1} = {flag}")

Justifikasi ambang.
- random -> PSI ≈ 0 di semua kolom, KS p tinggi -> tidak ada drift. Benar: batch acak i.i.d.,
  memang tak ada yang perlu dideteksi. Inilah alasan tutorial *terpaksa* menyuntik drift buatan.
- spatial -> KS p ≈ 0 (memicu jika hanya pakai KS!) tetapi PSI umumnya <0,2 -> kriteria gabungan
  menilainya moderat, belum layak retrain. Ini menunjukkan bahaya KS-only: ia akan retrain terus
  padahal pergeseran kecil.
- density -> PSI besar pada `density_ratio` (≈8) dan `n_distinct_types` (≈2,3–2,7), lalu `risk_score`
  (label) ikut bergeser -> drift nyata & layak retrain.
- Pemilihan angka: `PSI>0,2` mengikuti konvensi industri (batas "signifikan"); `α=0,01` (bukan 0,05)
  dibuat lebih ketat justru karena sampel besar membuat KS mudah signifikan. Trade-off: ambang lebih
  ketat -> hemat komputasi tapi berisiko telat mendeteksi; kombinasi KSdanPSI menyeimbangkan
  keduanya , butuh signifikansi *dan* magnitudo.

# 8. Checkpoint Retraining & Model Versioning

Rangkaian siklus continual learning:
1. Champion awal (v0) dilatih dari batch pertama (area paling sepi).
2. Tiap batch baru -> `detect_drift` terhadap data training kumulatif saat ini.
3. Tak ada drift -> skip retrain, batch cukup ditambahkan ke pool.
4. Ada drift -> latih kandidat dari seluruh data kumulatif, evaluasi di `holdout_df` yang sama.
5. Champion vs Challenger dengan promotion margin: kandidat dipromosikan hanya jika
   `MAE_kandidat ≤ MAE_champion − MARGIN`. Margin (0,10) menahan *model churn* akibat perbaikan
   yang terlalu kecil untuk sepadan dengan risiko mengganti model produksi.
6. Setiap versi (dipromosikan atau tidak) disimpan sebagai checkpoint & dicatat di registry.

**Dua perbaikan dibanding versi sebelumnya:**

- **Assembler ikut di-*fit* ulang di setiap retrain**, hanya pada `cumulative_train` saat itu.
  Ini menutup kebocoran lintas-batch yang sebelumnya ada: dulu model v0 , yang seharusnya hanya
  tahu area pilot , tetap menerima `density_ratio` dan `n_crimes` hasil agregasi HO1 atas
  **seluruh kota**, termasuk batch yang belum "tiba". Sekarang setiap versi hanya melihat data
  yang benar-benar sudah tersedia pada checkpoint tersebut, sehingga narasi drift dan perbaikan
  MAE-nya sah.
- **Checkpoint disimpan sebagai *bundle* `{model + assembler + daftar fitur + versi library}`**,
  bukan objek model telanjang. Model tanpa assembler-nya tidak dapat dipakai ulang, karena fitur
  agregatnya bergantung pada statistik hasil fit. Registry mencatat path, ukuran, dan SHA-256
  tiap checkpoint, ditulis *atomically* setiap kali ada entri baru.

Model yang dipakai konsisten dengan Bagian 4: HistGradientBoosting, 14 fitur leak-safe.

In [ ]:
MIRROR_TO_DRIVE = False
DRIVE_DIR       = "/content/drive/MyDrive/SISTECH/HO2_models"

ARTIFACT_DIR  = "models"
REGISTRY_PATH = os.path.join(ARTIFACT_DIR, "registry.json")
CHAMPION_PATH = os.path.join(ARTIFACT_DIR, "champion.json")
PROMOTE_MARGIN = 0.10
CL_ASSEMBLER_KW = dict(agg_source="refit", use_te=False)   

os.makedirs(ARTIFACT_DIR, exist_ok=True)


def _sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def _write_json(path, obj):
    """Tulis atomik: file sementara -> fsync -> replace. Registry tidak pernah setengah jadi."""
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)
    if MIRROR_TO_DRIVE:
        os.makedirs(DRIVE_DIR, exist_ok=True)
        shutil.copy2(path, os.path.join(DRIVE_DIR, os.path.basename(path)))


def load_registry():
    if os.path.exists(REGISTRY_PATH):
        with open(REGISTRY_PATH, encoding="utf-8") as f:
            return json.load(f)
    return []


def log_registry(entry):
    """Tambah entri, lalu PERSIST segera ke registry.json + registry.csv."""
    registry.append(entry)
    _write_json(REGISTRY_PATH, registry)
    pd.json_normalize(registry).to_csv(os.path.join(ARTIFACT_DIR, "registry.csv"), index=False)
    return entry


def set_champion(entry):
    _write_json(CHAMPION_PATH, {
        "version": entry["version"], "checkpoint_path": entry["checkpoint_path"],
        "checkpoint_sha256": entry["checkpoint_sha256"], "metrics": entry["metrics"],
        "batch_index": entry["batch_index"], "feature_names": entry["feature_names"],
        "updated_utc": datetime.now(timezone.utc).isoformat()})


def train_model(data_subset):
    """Assembler DI-FIT ULANG pada data yang tersedia di checkpoint ini (anti-leak lintas-batch)."""
    a = FeatureAssembler(**CL_ASSEMBLER_KW)
    X = a.fit_transform(data_subset)
    m = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.08, random_state=42)
    m.fit(X, data_subset[TARGET_COL].values)
    return m, a


def evaluate_model(model, assembler, data_eval):
    y = data_eval[TARGET_COL].values
    p = model.predict(assembler.transform(data_eval))
    band = pd.cut(y, [0, 20, 40, 60, 80, 100])
    mae_band = pd.Series(np.abs(y - p)).groupby(band, observed=False).mean()
    return {"MAE": float(mean_absolute_error(y, p)),
            "RMSE": float(np.sqrt(mean_squared_error(y, p))),
            "R2": float(r2_score(y, p)),
            "MAE_per_band": {str(k): (None if pd.isna(v) else float(v)) for k, v in mae_band.items()}}


def save_checkpoint(model, assembler, version, meta):
    """Simpan BUNDLE: model tanpa assembler-nya tidak bisa dipakai ulang."""
    path = os.path.join(ARTIFACT_DIR, f"model_v{version}.joblib")
    joblib.dump({"version": version, "model": model, "assembler": assembler,
                 "feature_names": assembler.feature_names_, "target": TARGET_COL,
                 "assembler_config": assembler.config(), "meta": meta,
                 "created_utc": datetime.now(timezone.utc).isoformat(),
                 "env": {"python": platform.python_version(), "sklearn": sklearn.__version__,
                         "pandas": pd.__version__, "numpy": np.__version__}}, path)
    if MIRROR_TO_DRIVE:
        os.makedirs(DRIVE_DIR, exist_ok=True)
        shutil.copy2(path, os.path.join(DRIVE_DIR, os.path.basename(path)))
    return path, _sha256(path), os.path.getsize(path)


def registry_entry(version, batch_index, train_size, model, assembler, metrics,
                   decision, drift_detected, drift_report=None, champion_before=None):
    ckpt, sha, size = (None, None, None)
    if model is not None:
        ckpt, sha, size = save_checkpoint(model, assembler, version,
                                          {"batch_index": batch_index, "decision": decision})
    return {
        "version": version,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "batch_index": batch_index,
        "trained_on_batches": list(range(batch_index + 1)) if model is not None else None,
        "train_size": int(train_size),
        "drift_detected": drift_detected,
        "drift_report": drift_report,
        "metrics": metrics,
        "champion_metrics_before": champion_before,
        "decision": decision,
        "promotion_rule": f"MAE_kandidat <= MAE_champion - {PROMOTE_MARGIN}",
        "model_type": type(model).__name__ if model is not None else None,
        "model_params": model.get_params() if model is not None else None,
        "assembler_config": assembler.config() if assembler is not None else None,
        "feature_names": assembler.feature_names_ if assembler is not None else None,
        "n_features": len(assembler.feature_names_) if assembler is not None else None,
        "checkpoint_path": ckpt, "checkpoint_sha256": sha, "checkpoint_bytes": size,
    }


registry = []          # mulai bersih; log_registry() menulis ke disk setiap kali dipanggil
print(f"Artefak -> {os.path.abspath(ARTIFACT_DIR)}")
print(f"Mirror ke Drive: {MIRROR_TO_DRIVE}" + (f" ({DRIVE_DIR})" if MIRROR_TO_DRIVE else ""))

In [ ]:
cumulative_train = batches[0].copy()
champion_model, champion_asm = train_model(cumulative_train)
champion_metrics = evaluate_model(champion_model, champion_asm, holdout_df)
version = 0

entry = registry_entry(version=0, batch_index=0, train_size=len(cumulative_train),
                       model=champion_model, assembler=champion_asm,
                       metrics=champion_metrics, decision="initial_champion",
                       drift_detected=None)
log_registry(entry); set_champion(entry)

print(f"[v0] champion dari batch 0 (n={len(cumulative_train):,}) | "
      f"MAE holdout={champion_metrics['MAE']:.3f} R2={champion_metrics['R2']:.3f}")
print(f"     assembler di-fit dari {champion_asm.n_fit_cells_:,} sel yang terlihat di batch 0 saja")
print(f"     checkpoint -> {entry['checkpoint_path']} "
      f"({entry['checkpoint_bytes']/1024:.0f} KB, sha256 {entry['checkpoint_sha256'][:12]}...)")

In [ ]:
for i in range(1, N_BATCHES):
    new_batch = batches[i]
    drift_detected, drift_report = detect_drift(cumulative_train, new_batch)
    print(f"\n=== Batch {i} tiba (n={len(new_batch):,}) | drift={drift_detected} ===")
    for col, dd in drift_report.items():
        print(f"   {col:16s} KS_p={dd['ks_p']:.3g}  PSI={dd['psi']:.3f}  drift={dd['drift']}")

    cumulative_train = pd.concat([cumulative_train, new_batch], ignore_index=True)

    if not drift_detected:
        log_registry(registry_entry(version=None, batch_index=i, train_size=len(cumulative_train),
                                    model=None, assembler=None, metrics=None,
                                    decision="skip_retrain_no_drift",
                                    drift_detected=False, drift_report=drift_report))
        print("   -> skip retrain (tidak ada drift signifikan); batch hanya ditambahkan ke pool")
        continue

    # retrain: model DAN assembler dilatih ulang dari data kumulatif yang tersedia saat ini
    candidate_model, candidate_asm = train_model(cumulative_train)
    candidate_metrics = evaluate_model(candidate_model, candidate_asm, holdout_df)
    version += 1

    improved = candidate_metrics["MAE"] <= champion_metrics["MAE"] - PROMOTE_MARGIN
    decision = "promoted" if improved else "kept_champion"
    entry = registry_entry(version=version, batch_index=i, train_size=len(cumulative_train),
                           model=candidate_model, assembler=candidate_asm,
                           metrics=candidate_metrics, decision=decision,
                           drift_detected=True, drift_report=drift_report,
                           champion_before=champion_metrics)
    log_registry(entry)

    delta = champion_metrics["MAE"] - candidate_metrics["MAE"]
    print(f"   [v{version}] kandidat MAE={candidate_metrics['MAE']:.3f} R2={candidate_metrics['R2']:.3f} "
          f"vs champion MAE={champion_metrics['MAE']:.3f} | Δ={delta:+.3f} "
          f"(margin {PROMOTE_MARGIN}) -> {decision}")
    print(f"        checkpoint tersimpan: {entry['checkpoint_path']} "
          f"(sha256 {entry['checkpoint_sha256'][:12]}...)")

    if improved:
        champion_model, champion_asm, champion_metrics = candidate_model, candidate_asm, candidate_metrics
        set_champion(entry)
        print(f"        champion.json diperbarui -> v{version}")

print(f"\n=== Selesai === Champion akhir: MAE={champion_metrics['MAE']:.3f} "
      f"R2={champion_metrics['R2']:.3f}")
print(f"Registry: {len(registry)} entri tersimpan di {REGISTRY_PATH}")

# 9. Model Registry & Checkpoints

Registry adalah satu-satunya sumber kebenaran tentang "model mana yang sedang produksi dan mengapa".
Tiga sel berikut memastikan ia benar-benar **tersimpan**, bukan sekadar hidup di memori session:

1. **Isi registry ditampilkan sekaligus dicetak utuh sebagai JSON**, sehingga tetap terbaca di
   file `.ipynb` yang dikumpulkan meskipun folder `models/` tidak ikut terkirim.
2. **Uji muat-ulang**: champion dibaca kembali dari disk (bundle model + assembler), diprediksikan
   ke holdout, lalu MAE-nya dibandingkan dengan yang tercatat di registry. Kalau tidak identik,
   sel ini gagal , jadi klaim "tersimpan dan dapat dipakai ulang" terbukti, bukan diasumsikan.
3. **Ekspor artefak**: seluruh folder `models/` dikemas menjadi satu `.zip` yang diunduh, sehingga
   registry dan checkpoint ikut dikumpulkan bersama notebook.

Isi setiap entri registry: versi, timestamp UTC, batch asal, ukuran data latih, laporan drift
(KS p-value & PSI per kolom), metrik holdout (MAE/RMSE/R² + MAE per band risiko), metrik champion
sebelumnya, keputusan promosi beserta aturannya, tipe & hyperparameter model, konfigurasi assembler,
daftar fitur, serta path + ukuran + SHA-256 checkpoint.

In [ ]:
print(f"Isi folder {ARTIFACT_DIR}/:")
for f in sorted(os.listdir(ARTIFACT_DIR)):
    print(f"  - {f:24s} {os.path.getsize(os.path.join(ARTIFACT_DIR, f))/1024:8.1f} KB")

registry_on_disk = load_registry()          # dibaca ULANG dari file, bukan dari variabel memori
print(f"\nregistry.json memuat {len(registry_on_disk)} entri "
      f"(variabel di memori: {len(registry)}) -> cocok: {len(registry_on_disk) == len(registry)}")

registry_df = pd.json_normalize(registry_on_disk)
cols_show = [c for c in ["version","batch_index","train_size","drift_detected","decision",
                         "metrics.MAE","metrics.R2","champion_metrics_before.MAE",
                         "n_features","checkpoint_path"] if c in registry_df.columns]
registry_df[cols_show].round(3)

In [ ]:
champ = json.load(open(CHAMPION_PATH, encoding="utf-8"))
print(f"champion.json -> v{champ['version']} | {champ['checkpoint_path']}")

assert _sha256(champ["checkpoint_path"]) == champ["checkpoint_sha256"], "checkpoint korup!"
bundle = joblib.load(champ["checkpoint_path"])          

y_ho = holdout_df[TARGET_COL].values
p_ho = bundle["model"].predict(bundle["assembler"].transform(holdout_df))
mae_reload = float(mean_absolute_error(y_ho, p_ho))

print(f"MAE dari checkpoint yang dimuat ulang : {mae_reload:.6f}")
print(f"MAE yang tercatat di registry         : {champ['metrics']['MAE']:.6f}")
assert abs(mae_reload - champ["metrics"]["MAE"]) < 1e-9, "metrik registry tidak reproducible!"
print("\nREGISTRY OK , checkpoint utuh (sha256 cocok) & metriknya reproducible dari disk.")
print(f"dilatih dengan {bundle['env']} pada {bundle['created_utc']}")
print(f"fitur ({len(bundle['feature_names'])}): {bundle['feature_names']}")
print(f"konfigurasi assembler: {bundle['assembler_config']}")

# --- 3) Registry utuh dicetak, agar tetap ada di dalam .ipynb yang dikumpulkan ---
print("\n" + "="*70 + "\nISI LENGKAP registry.json\n" + "="*70)
print(json.dumps(registry_on_disk, indent=2, ensure_ascii=False, default=str))

In [ ]:
zip_path = shutil.make_archive("HO2_model_artifacts", "zip", ARTIFACT_DIR)
print(f"Artefak dikemas: {zip_path} ({os.path.getsize(zip_path)/1024/1024:.2f} MB)")
print("Isi:", ", ".join(sorted(os.listdir(ARTIFACT_DIR))))

try:                                    
    from google.colab import files
    files.download(zip_path)
    print("-> unduhan dimulai. Kumpulkan HO2_model_artifacts.zip bersama notebook & README.")
except Exception as exc:
    print(f"-> bukan Colab ({type(exc).__name__}); ambil file zip di atas secara manual.")

In [ ]:
hist = [e for e in registry_on_disk if e.get("metrics")]
vv   = [e["batch_index"] for e in hist]
mae  = [e["metrics"]["MAE"] for e in hist]
dec  = [e["decision"] for e in hist]

champ_line, cur = [], None
for e in hist:                                     
    if e["decision"] in ("initial_champion", "promoted"): cur = e["metrics"]["MAE"]
    champ_line.append(cur)

mae_percell = float(baseline_results_df.loc[
    baseline_results_df["model"] == "Baseline: per-Sel", "MAE"].iloc[0])

plt.figure(figsize=(8,4))
plt.plot(vv, mae, "-o", color="#4C72B0", label="MAE kandidat")
plt.step(vv, champ_line, where="post", color="#C44E52", label="MAE champion (produksi)")
for x, y, dd in zip(vv, mae, dec):
    plt.annotate(dd.replace("_", "\n"), (x, y), textcoords="offset points", xytext=(0, 8),
                 ha="center", fontsize=8)
plt.axhline(mae_percell, ls="--", color="gray", label=f"Baseline per-Sel ({mae_percell:.2f})")
plt.xlabel("Batch index"); plt.ylabel("MAE holdout")
plt.title("Model Journey , Continual Learning (fitur leak-safe)")
plt.legend(); plt.tight_layout(); plt.show()

# 10. Ringkasan & Refleksi

Hasil akhir:

- Model terbaik statis: HistGradientBoosting dengan 14 fitur leak-safe, mengalahkan baseline
  terkuat (per-Sel). Pengungkit terbesar tetap *feature assembly* (memakai kembali agregat spasial
  HO1), disusul pemilihan kelas model non-linear.
- Model journey skema coverage-expansion: drift terdeteksi dari data nyata di setiap batch,
  retrain melatih ulang model **dan** assembler, dan promotion margin menahan kandidat yang
  perbaikannya terlalu kecil. Seluruh keputusan dapat ditelusuri di `models/registry.json`.

Keterbatasan:

- `mean_severity`, `max_severity`, `pct_violent`, `n_distinct_types` adalah konstanta per-sel yang
  di HO1 diringkas dari catatan kejadian, kolom severity per-kejadian sudah tidak ada di
  `features_labels.csv`, sehingga rekonstruksi train-only tidak mungkin sepenuhnya. Harga membuang
  keempatnya sudah diukur (baris "tanpa agregat severity).
- `crime_count` tidak tersedia saat serving untuk slot yang belum terjadi. Varian tanpa
  `crime_count` menunjukkan performa yang realistis untuk deployment.
- Sumbu waktu asli hilang akibat agregasi HO1, sehingga batch memakai skema coverage-expansion
  (simulasi rollout), bukan kronologi sungguhan.
- Plateau R² ~0,3 berasal dari komponen *spatial/temporal decay* label HO1 yang tak terekonstruksi
  dari agregat per-sel.
- Checkpoint memakai `joblib` atas kelas `FeatureAssembler` yang didefinisikan di notebook; untuk
  produksi kelas ini perlu dipindahkan ke modul `.py` agar bundle dapat dimuat di proses lain.

In [ ]:
def id_num(x, nd=3):
    return "—" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{x:.{nd}f}".replace(".", ",")

def md_table(df, cols, headers, align):
    out = ["| " + " | ".join(headers) + " |", "| " + " | ".join(align) + " |"]
    for _, r in df.iterrows():
        out.append("| " + " | ".join(
            r[c] if isinstance(r[c], str) else id_num(r[c]) for c in cols) + " |")
    return "\n".join(out)

print("### Tabel 1 , Baseline non-ML\n")
print(md_table(baseline_results_df, ["model","MAE","RMSE","R2"],
               ["Baseline","MAE","RMSE","R²"], [":--","--:","--:","--:"]))

print("\n\n### Tabel 2 , Model (semua leak-safe kecuali baris [BOCOR])\n")
mr = pd.DataFrame(model_results)
mr["n_fitur"] = mr["n_fitur"].astype(str)
print(md_table(mr, ["model","n_fitur","MAE","RMSE","R2"],
               ["Model","#Fitur","MAE","RMSE","R²"], [":--","--:","--:","--:","--:"]))

print("\n\n### Tabel 3 , Model journey (dari registry.json)\n")
jr = pd.DataFrame([{
    "batch": str(e["batch_index"]),
    "versi": "—" if e["version"] is None else f"v{e['version']}",
    "drift": {True: "Ya", False: "Tidak", None: "—"}[e["drift_detected"]],
    "MAE": None if not e["metrics"] else e["metrics"]["MAE"],
    "R2":  None if not e["metrics"] else e["metrics"]["R2"],
    "keputusan": e["decision"],
} for e in registry_on_disk])
print(md_table(jr, ["batch","versi","drift","MAE","R2","keputusan"],
               ["Batch","Versi","Drift?","MAE kandidat","R²","Keputusan"],
               [":-:",":-:",":-:","--:","--:",":--"]))

print("\n\n### Tabel 4 , MAE per band risiko (champion akhir)\n")
print("| Band Risk Score | MAE |\n| :-- | --: |")
for k, v in champ["metrics"]["MAE_per_band"].items():
    print(f"| {k} | {id_num(v, 2)} |")

print("\n\n### Angka kunci\n")
print(f"- Champion akhir: MAE {id_num(champion_metrics['MAE'])} · "
      f"R² {id_num(champion_metrics['R2'])} (v{champ['version']}, batch {champ['batch_index']})")
print(f"- Baseline pembanding (per-Sel): MAE {id_num(mae_percell)}")
print(f"- Optimisme akibat kebocoran versi lama: {id_num(gap)} poin MAE")
print(f"- Registry: {len(registry_on_disk)} entri, "
      f"{sum(1 for e in registry_on_disk if e['checkpoint_path'])} checkpoint di {ARTIFACT_DIR}/")